# Tamaños de efecto (R2-5) y cobertura muestral (R2-9)

Notebook independiente para responder al Revisor 2 (comentarios 5 y 9). Para R2-5, reajusta el pipeline completo (K=8) y calcula eta-cuadrado y d de Cohen para PUNT_GLOBAL por clúster, emparejando por rango de puntaje medio con la Tabla 7 real (no por número de etiqueta, que es arbitrario entre corridas de K-Means). Para R2-9, cuenta identificadores de estudiante repetidos.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/tamanos_efecto_cobertura/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/tamanos_efecto_cobertura'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. R2-9 — Cobertura muestral: ¿cuántos estudiantes se repiten?

No requiere reajustar el pipeline — es un conteo directo sobre `ESTU_CONSECUTIVO`.

In [ ]:
conteo = df_filtrado_full['ESTU_CONSECUTIVO'].value_counts() if 'ESTU_CONSECUTIVO' in df_filtrado_full.columns else None
if conteo is not None:
    n_repetidos = int((conteo > 1).sum())
    n_registros_de_repetidos = int(conteo[conteo > 1].sum())
    n_unicos = int(len(conteo))
    print(f"Identificadores únicos: {n_unicos:,} de {len(df_filtrado_full):,} registros")
    print(f"Registros que comparten identificador con otro: "
          f"{n_registros_de_repetidos:,} ({n_registros_de_repetidos/len(df_filtrado_full)*100:.2f}%)")
    print("\nValor esperado (manuscrito): 1,051 de 452,020 (0.23%), 450,969 identificadores únicos.")
else:
    print("ESTU_CONSECUTIVO no está en las columnas conservadas — revisa preprocesar_saber_pro().")

## 4. R2-5 — Partición fresca K=8 y verificación de tamaños contra la Tabla 7

⏱️ Este es el paso más pesado: un UMAP fit (80,000) + transform (452,020) — puede tardar 10-20 minutos.

In [ ]:
TABLE7_SORTED = [  # (tamaño, media), ordenado por media ascendente — Tabla 7 real del manuscrito
    (54623, 135.92), (60240, 136.30), (68047, 140.92), (69891, 145.69),
    (63929, 148.48), (66104, 150.21), (62257, 161.98), (6929, 205.28),
]

SEED_R25 = 42
rng_fit = np.random.default_rng(seed=SEED_R25)
idx_fit_r25 = rng_fit.choice(n_total, size=80_000, replace=False)
print("Ajustando UMAP sobre 80,000 filas y transformando las 452,020...")
reducer_r25 = umap_cpu.UMAP(n_components=2, random_state=SEED_R25, n_neighbors=10,
                             low_memory=True, n_jobs=-1)
reducer_r25.fit(X_full[idx_fit_r25])
emb_full_r25 = reducer_r25.transform(X_full)
km_r25 = MiniBatchKMeans(n_clusters=8, random_state=SEED_R25, n_init="auto", batch_size=10_000)
labels_run = km_r25.fit_predict(emb_full_r25)

df_filtrado_full['_label_run'] = labels_run
df_filtrado_full['PUNT_GLOBAL_calc'] = df_filtrado_full[
    ['MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT', 'MOD_COMPETEN_CIUDADA_PUNT',
     'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']].mean(axis=1)
means_run = df_filtrado_full.groupby('_label_run')['PUNT_GLOBAL_calc'].mean().sort_values()
sizes_run = df_filtrado_full['_label_run'].value_counts().reindex(means_run.index)

print("Verificación de tamaños por rango (corrida fresca vs. Tabla 7 real):")
for i, (lbl, n_run) in enumerate(zip(means_run.index, sizes_run)):
    n_t7, mean_t7 = TABLE7_SORTED[i]
    diff_pct = 100 * abs(n_run - n_t7) / n_t7
    print(f"  rango {i}: etiqueta={lbl}  n_corrida={n_run}  n_Tabla7={n_t7}  diff={diff_pct:.1f}%")

## 5. Eta-cuadrado y d de Cohen (por pares, 28 combinaciones de 8 clústeres)

In [ ]:
import itertools

y_glob = df_filtrado_full['PUNT_GLOBAL_calc'].values
grand_mean = y_glob.mean()
ss_total = ((y_glob - grand_mean) ** 2).sum()
ss_between = sum(len(g) * (g['PUNT_GLOBAL_calc'].mean() - grand_mean) ** 2
                 for _, g in df_filtrado_full.groupby('_label_run'))
eta_sq = ss_between / ss_total
print(f"eta-cuadrado (PUNT_GLOBAL por clúster) = {eta_sq:.3f}")

groups = {lbl: g['PUNT_GLOBAL_calc'].values for lbl, g in df_filtrado_full.groupby('_label_run')}
ds = []
for a, b in itertools.combinations(groups.keys(), 2):
    ga, gb = groups[a], groups[b]
    na, nb = len(ga), len(gb)
    pooled_sd = np.sqrt(((na - 1) * ga.var(ddof=1) + (nb - 1) * gb.var(ddof=1)) / (na + nb - 2))
    ds.append(abs(ga.mean() - gb.mean()) / pooled_sd)
ds = np.array(ds)

print(f"Cohen's d por pares (n={len(ds)} pares): min={ds.min():.2f}  "
      f"mediana={np.median(ds):.2f}  max={ds.max():.2f}")
print("\nValores esperados (manuscrito): eta²=0.195, d entre 0.05 y 3.53 (mediana=0.59).")

json.dump({'eta_cuadrado': float(eta_sq), 'cohens_d_min': float(ds.min()),
           'cohens_d_mediana': float(np.median(ds)), 'cohens_d_max': float(ds.max())},
          open(os.path.join(OUT_DIR, 'resultados.json'), 'w'), indent=2)